# 01 - Modelo de ejecución: del código al job

## Conexión con la teoría

Este notebook aterriza transformaciones, acciones, lazy evaluation, planes lógico y físico,
dependencias narrow y wide, shuffle, lineage, cache y checkpoint.

> Antes de ejecutar cada bloque: **predice si habrá un job y si aparecerá `Exchange` en el plan**.

In [4]:
import os, shutil, subprocess

os.environ["JAVA_HOME"] = "/Users/beatrizsuch/java/temurin-17/Contents/Home"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:' + os.environ["PATH"]

print("JAVA_HOME =", os.environ.get("JAVA_HOME"))
print("java path =", shutil.which("java"))
subprocess.run(["java", "-version"], check=True)

JAVA_HOME = /Users/beatrizsuch/java/temurin-17/Contents/Home
java path = /Users/beatrizsuch/java/temurin-17/Contents/Home/bin/java


openjdk version "17.0.21" 2026-10-20
OpenJDK Runtime Environment Temurin-17.0.21+9 (build 17.0.21+9)
OpenJDK 64-Bit Server VM Temurin-17.0.21+9 (build 17.0.21+9, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

In [5]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("pontia-computacion-distribuida")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} | {spark.sparkContext.master}")
print(f"Paralelismo por defecto: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark 4.3.0 | local[*]
Paralelismo por defecto: 8
Spark UI: http://192.168.1.68:4040


## 1. Datos repartidos en cuatro particiones

In [6]:
events = (
    spark.range(0, 100_000, numPartitions=4)
    .withColumn("user_id", (F.col("id") % 5_000).cast("long"))
    .withColumn("category", F.concat(F.lit("cat_"), (F.col("id") % 20)))
    .withColumn("amount", ((F.col("id") % 100) + 1).cast("double"))
)

print("Particiones:", events.rdd.getNumPartitions())
events.show(5)

Particiones: 4
+---+-------+--------+------+
| id|user_id|category|amount|
+---+-------+--------+------+
|  0|      0|   cat_0|   1.0|
|  1|      1|   cat_1|   2.0|
|  2|      2|   cat_2|   3.0|
|  3|      3|   cat_3|   4.0|
|  4|      4|   cat_4|   5.0|
+---+-------+--------+------+
only showing top 5 rows


## 2. Transformaciones lazy

Python ejecuta estas líneas y construye objetos DataFrame. Spark todavía no recorre las
100.000 filas para producir el resultado.

In [7]:
prepared = (
    events
    .filter(F.col("amount") >= 50)
    .select("user_id", "category", "amount")
    .withColumn("amount_with_tax", F.round(F.col("amount") * 1.21, 2))
)

print("Pipeline definido: todavía no hemos solicitado filas.")

Pipeline definido: todavía no hemos solicitado filas.


## 3. El plan permite formular una hipótesis antes de ejecutar

In [8]:
prepared.explain(mode="formatted")

== Physical Plan ==
* Project (4)
+- * Project (3)
   +- * Filter (2)
      +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 100000, step=1, splits=Some(4))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : (cast(((id#0L % 100) + 1) as double) >= 50.0)

(3) Project [codegen id : 1]
Output [3]: [(id#0L % 5000) AS user_id#1L, concat(cat_, cast((id#0L % 20) as string)) AS category#2, cast(((id#0L % 100) + 1) as double) AS amount#3]
Input [1]: [id#0L]

(4) Project [codegen id : 1]
Output [4]: [user_id#1L, category#2, amount#3, round((amount#3 * 1.21), 2) AS amount_with_tax#17]
Input [3]: [user_id#1L, category#2, amount#3]




Busca `Scan`, `Filter` y `Project`. Son operaciones narrow: cada partición puede continuar
trabajando con sus propios datos. Ahora introducimos una dependencia wide.

In [9]:
summary = (
    prepared
    .groupBy("category")
    .agg(
        F.count("*").alias("events"),
        F.round(F.sum("amount_with_tax"), 2).alias("revenue"),
    )
)

summary.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 100000, step=1, splits=Some(4))

(2) Filter
Input [1]: [id#0L]
Condition : (cast(((id#0L % 100) + 1) as double) >= 50.0)

(3) Project
Output [2]: [concat(cat_, cast((id#0L % 20) as string)) AS category#2, round((cast(((id#0L % 100) + 1) as double) * 1.21), 2) AS amount_with_tax#17]
Input [1]: [id#0L]

(4) HashAggregate
Input [2]: [category#2, amount_with_tax#17]
Keys [1]: [category#2]
Functions [2]: [partial_count(1), partial_sum(amount_with_tax#17)]
Aggregate Attributes [2]: [count#26L, sum#27]
Results [3]: [category#2, count#28L, sum#29]

(5) Exchange
Input [3]: [category#2, count#28L, sum#29]
Arguments: hashpartitioning(category#2, 4), ENSURE_REQUIREMENTS, [plan_id=55]

(6) HashAggregate
Input [3]: [category#2, count#28L, sum#29]
Keys [1]: [catego

### Pregunta

¿Dónde aparece `Exchange`? Ese operador representa una redistribución: las filas de una
misma categoría deben terminar juntas para completar la agregación.

## 4. Una acción crea trabajo real

In [10]:
summary.orderBy(F.desc("revenue")).show(10)

+--------+------+--------+
|category|events| revenue|
+--------+------+--------+
|  cat_19|  3000|290400.0|
|  cat_18|  3000|286770.0|
|  cat_17|  3000|283140.0|
|  cat_16|  3000|279510.0|
|  cat_15|  3000|275880.0|
|  cat_14|  3000|272250.0|
|  cat_13|  3000|268620.0|
|  cat_12|  3000|264990.0|
|  cat_11|  3000|261360.0|
|  cat_10|  3000|257730.0|
+--------+------+--------+
only showing top 10 rows


La acción genera un job. El DAG Scheduler lo divide en stages alrededor del shuffle y cada
stage crea tasks para sus particiones. Abre Spark UI y localiza Jobs, Stages y SQL/DataFrame.

## 5. Dos acciones pueden recorrer dos veces el lineage

In [11]:
summary.count()
summary.orderBy("category").show(5)

+--------+------+--------+
|category|events| revenue|
+--------+------+--------+
|   cat_0|  2000|171820.0|
|   cat_1|  2000|174240.0|
|  cat_10|  3000|257730.0|
|  cat_11|  3000|261360.0|
|  cat_12|  3000|264990.0|
+--------+------+--------+
only showing top 5 rows


Spark no conserva automáticamente todos los resultados. Comprueba en la UI cuántos jobs se
han creado. Ahora materializaremos el resultado para reutilizarlo.

In [12]:
cached_summary = summary.cache()
cached_summary.count()  # primera acción: materializa la cache
cached_summary.orderBy("category").show(5)  # puede reutilizarla

print("Storage level:", cached_summary.storageLevel)
cached_summary.unpersist()

+--------+------+--------+
|category|events| revenue|
+--------+------+--------+
|   cat_0|  2000|171820.0|
|   cat_1|  2000|174240.0|
|  cat_10|  3000|257730.0|
|  cat_11|  3000|261360.0|
|  cat_12|  3000|264990.0|
+--------+------+--------+
only showing top 5 rows
Storage level: Disk Memory Deserialized 1x Replicated


DataFrame[category: string, events: bigint, revenue: double]

## 6. Checkpoint: materializar y cortar el lineage

`cache` acelera la reutilización, pero mantiene las dependencias. `checkpoint` crea un nuevo
punto de partida. Escribe en almacenamiento fiable y resulta útil en planes largos o iterativos.

In [13]:
checkpoint_dir = DATA_DIR / "checkpoints"
spark.sparkContext.setCheckpointDir(str(checkpoint_dir))

checkpointed = summary.checkpoint(eager=True)
checkpointed.explain(mode="formatted")

== Physical Plan ==
* Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [3]: [category#2, events#18L, revenue#19]
Arguments: [category#2, events#18L, revenue#19], MapPartitionsRDD[63] at checkpoint at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)




## Preguntas de cierre

1. ¿Qué operaciones eran narrow?
2. ¿Cuál introdujo la dependencia wide?
3. ¿Por qué `cache()` necesita una acción para materializarse?
4. ¿Qué diferencia observas entre el plan anterior y el plan tras el checkpoint?

In [14]:
spark.stop()